In [2]:
import pandas as pd

recipes = pd.read_parquet('../data/recipes_sample.parquet')
ids = set(recipes["RecipeId"])

In [4]:
len(ids)

4983

## Filter reviews to the sampled recipes

In [5]:
import kagglehub
import tiktoken
from pathlib import Path

path = Path(kagglehub.dataset_download("irkaal/foodcom-recipes-and-reviews"))
reviews = pd.read_parquet(path / "reviews.parquet")
print(f"all reviews: {len(reviews):,}")

r = reviews[reviews["RecipeId"].isin(ids)].copy()
r["Review"] = r["Review"].astype("string")
r = r[r["Review"].str.strip().str.len() > 0]           # drop null / blank text

enc = tiktoken.encoding_for_model("text-embedding-3-small")
r["token_count"] = r["Review"].map(lambda t: len(enc.encode(t)))
r = r[r["token_count"] < 8192].reset_index(drop=True)   # embedding context limit

print(f"reviews to embed: {len(r):,}  covering {r['RecipeId'].nunique():,} recipes")
print(f"~{r['token_count'].sum():,} tokens")

/Users/nico/serenis-health/ai-bootcamp/ragu/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


all reviews: 1,401,982
reviews to embed: 13,739  covering 2,589 recipes
~898,278 tokens


## Build the reviews collection

In [7]:
import math
from dotenv import load_dotenv
load_dotenv("../.env")

import openai
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct, PayloadSchemaType

qdrant = QdrantClient(url="http://localhost:6333")
COLLECTION = "Recipes-reviews-collection-01"

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)
qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config={"text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)},
)
qdrant.create_payload_index(COLLECTION, field_name="RecipeId", field_schema=PayloadSchemaType.INTEGER)


def get_embeddings(texts, model="text-embedding-3-small"):
    resp = openai.embeddings.create(input=texts, model=model)
    return [d.embedding for d in sorted(resp.data, key=lambda d: d.index)]  # keep input order


BATCH = 1000
rows = r.to_dict(orient="records")
for start in range(0, len(rows), BATCH):
    batch = rows[start:start + BATCH]
    embs = get_embeddings([row["Review"] for row in batch])
    points = [
        PointStruct(
            id=int(row["ReviewId"]),
            vector={"text-embedding-3-small": emb},
            payload={
                "Review": row["Review"],
                "RecipeId": int(row["RecipeId"]),
                "Rating": None if math.isnan(row["Rating"]) else int(row["Rating"]),
            },
        )
        for row, emb in zip(batch, embs)
    ]
    qdrant.upsert(collection_name=COLLECTION, points=points, wait=True)
    print(f"upserted {min(start + BATCH, len(rows)):,} / {len(rows):,}")

print("points:", qdrant.get_collection(COLLECTION).points_count)

upserted 1,000 / 13,739
upserted 2,000 / 13,739
upserted 3,000 / 13,739
upserted 4,000 / 13,739
upserted 5,000 / 13,739
upserted 6,000 / 13,739
upserted 7,000 / 13,739
upserted 8,000 / 13,739
upserted 9,000 / 13,739
upserted 10,000 / 13,739
upserted 11,000 / 13,739
upserted 12,000 / 13,739
upserted 13,000 / 13,739
upserted 13,739 / 13,739
points: 13739


## Verify the prefiltered join

In [8]:
from qdrant_client.models import Filter, FieldCondition, MatchAny


def search_reviews(query, recipe_ids, k=5):
    emb = openai.embeddings.create(input=query, model="text-embedding-3-small").data[0].embedding
    return qdrant.query_points(
        collection_name=COLLECTION,
        query=emb,
        using="text-embedding-3-small",
        query_filter=Filter(must=[FieldCondition(key="RecipeId", match=MatchAny(any=recipe_ids))]),
        limit=k,
    )


# prefilter to two specific recipes, then rank their reviews semantically
res = search_reviews("delicious, easy, family favorite", [52799, 108775])
for p in res.points:
    print(f"score={p.score:.3f}  RecipeId={p.payload['RecipeId']}  rating={p.payload['Rating']}  {p.payload['Review'][:80]}")

score=0.710  RecipeId=108775  rating=5  delicious and so easy to cook.. 5 stars
score=0.693  RecipeId=52799  rating=5  Super easy and delicious.
score=0.652  RecipeId=108775  rating=5  Very easy to make and the entire family enjoyed it.
score=0.635  RecipeId=108775  rating=5  Delicious and easy! My family really liked the coating.  Since we like our food 
score=0.610  RecipeId=108775  rating=5  Thanks for this delicious recipe!  I have made this many times (especially for c
